In [1]:
# import packages
import csv
import numpy as np
import pandas as pd

In [2]:
#Daftar file dan metadata
files = [
    ("smp_2021.csv", "SMP", 2021),
    ("smp_2022.csv", "SMP", 2022),
    ("smp_2023.csv", "SMP", 2023),
    ("smp_2024.csv", "SMP", 2024),
]

In [3]:
# List untuk menyimpan data
df_all = []

In [6]:
# Loop untuk membaca dan memproses tiap file
for file, jenjang, tahun in files:
    try:
        # Path ke file
        file_path = f"/content/sample_data/dataset_smp/{file}"

        # Baca CSV
        df = pd.read_csv(file_path)

        # Normalisasi nama kolom (ubah jika nama kolom berbeda)
        df = df.rename(columns={
            "Laki-laki (L) - Tingkat - VII": "L_VII",
            "Laki-laki (L) - Tingkat - VIII": "L_VIII",
            "Laki-laki (L) - Tingkat - IX": "L_IX",
            "Laki-laki (L) - Subjml": "L_Jumlah",
            "Perempuan (P) - Tingkat - VII": "P_VII",
            "Perempuan (P) - Tingkat - VIII": "P_VIII",
            "Perempuan (P) - Tingkat - IX": "P_IX",
            "Perempuan (P) - Subjml": "P_Jumlah",
        })

        # Tambah metadata Jenjang dan Tahun
        df["Jenjang"] = jenjang
        df["Tahun"] = tahun

        # Ubah ke format long
        df_long = pd.melt(
            df,
            id_vars=["Kecamatan", "Jenjang", "Tahun"],
            value_vars=["L_VII", "L_VIII", "L_IX", "L_Jumlah", "P_VII", "P_VIII", "P_IX", "P_Jumlah"],
            var_name="Kategori",
            value_name="Jumlah"
        )

        # Tambahkan kolom Jenis Kelamin dan Tingkat
        def extract_gender(k):
            return "Laki-laki" if k.startswith("L") else "Perempuan"

        def extract_grade(k):
            if "_IX" in k:
                return "IX"
            elif "_VIII" in k:
                return "VIII"
            elif "_VII" in k:
                return "VII"
            else:
                return "Total"

        df_long["Jenis Kelamin"] = df_long["Kategori"].apply(extract_gender)
        df_long["Tingkat"] = df_long["Kategori"].apply(extract_grade)

        # Hapus kolom kategori
        df_long = df_long.drop(columns="Kategori")

        # Simpan
        df_all.append(df_long)

    except Exception as e:
        print(f"Gagal memproses file {file}: {e}")

In [7]:
# Gabungkan semua jadi satu DataFrame besar
df_final = pd.concat(df_all, ignore_index=True)

In [8]:
# Tampilkan hasil
print(df_final.head())

                   Kecamatan Jenjang  Tahun  Jumlah Jenis Kelamin Tingkat
0    Kec. Teluk Betung Barat     SMP   2021       0     Laki-laki     VII
1  Kec. Teluk Betung Selatan     SMP   2021       0     Laki-laki     VII
2               Kec. Panjang     SMP   2021       2     Laki-laki     VII
3  Kec. Tanjung Karang Timur     SMP   2021       0     Laki-laki     VII
4    Kec. Teluk Betung Utara     SMP   2021       1     Laki-laki     VII


In [10]:
# Simpan ke CSV
df_final.to_csv("dataset_smp.csv", index=False)